# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [2]:
!ollama list

NAME                                   ID              SIZE      MODIFIED    
deepseek-r1:1.5b                       e0979632db5a    1.1 GB    7 hours ago    
llama3.2:1b                            baf6a787fdff    1.3 GB    7 hours ago    
nomic-embed-text:latest                0a109f422b47    274 MB    2 weeks ago    
artifish/llama3.2-uncensored:latest    c73bea26e004    2.2 GB    2 weeks ago    


In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [4]:
# Initialize and constants

load_dotenv(override=True)
# api_key = os.getenv('OPENAI_API_KEY')
ollama_base_url = os.getenv('OLLAMA_BASE_URL')

# if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
#     print("API key looks good so far")
# else:
#     print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'llama3.2:1b'
openai = OpenAI(base_url = ollama_base_url , api_key = MODEL)

In [5]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/1

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [6]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [7]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [8]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/

In [9]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [10]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about-me-and-about-nebula',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula'},
  {'type': 'news',
   'url': 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html'}]}

In [11]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [12]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama3.2:1b
Found 2 relevant links


{'links': [{'type': 'about',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'posts', 'url': 'https://edwarddonner.com/posts/'}]}

In [13]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2:1b
Found 3 relevant links


{'links': [{'type': 'endpoints', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'brand', 'url': 'https://github.com/huggingface'},
  {'type': 'blog', 'url': 'https://discuss.huggingface.co'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [20]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [21]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling llama3.2:1b
Found 2 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
zai-org/GLM-OCR
Updated
4 days ago
•
204k
•
753
Qwen/Qwen3-Coder-Next
Updated
4 days ago
•
53.5k
•
560
moonshotai/Kimi-K2.5
Updated
2 days ago
•
335k
•
1.81k
stepfun-ai/Step-3.5-Flash
Updated
about 8 hours ago
•
12k
•
501
circlestone-labs/Anima
Updated
7 days ago
•
60.6k
•
486
Browse 2M+ models
Spaces
Running
on
Zero
Featured
1.28k
Qwen3-TTS Demo
🎙
1.28k
Transform text into natural-sounding speech with custom voices
Running
on
A100
167
ACE-Step v1.5
🎵
167
Music Generation Foundation Model v1.5
Running
464
Demo Playground
⚡
464
Free p

In [28]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [29]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [30]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2:1b
Found 3 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nzai-org/GLM-OCR\nUpdated\n4 days ago\n•\n204k\n•\n753\nQwen/Qwen3-Coder-Next\nUpdated\n4 days ago\n•\n53.5k\n•\n560\nmoonshotai/Kimi-K2.5\nUpdated\n2 days ago\n•\n335k\n•\n1.81k\nstepfun-ai/Step-3.5-Flash\nUpdated\nabout 8 hours ago\n•\n12k\n•\n502\ncirclestone-labs/Anima\nUpdated\n7 days ago\n•\n60.6k\n•\n487\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n1.28k\nQwen3-TTS Demo\n🎙

In [37]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="llama3.2:1b",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [38]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2:1b
Found 4 relevant links


# Hugging Face: The AI Community Building the Future

## Introduction

Hugging Face is a leading platform for machine learning and artificial intelligence, where the machine learning community collaborates on models, datasets, and applications. Our mission is to facilitate the creation, discovery, and collaboration of machine learning projects, enabling developers and researchers to build innovative solutions.

## Products & Services

### Models

Our platform offers a vast collection of pre-trained models for various applications, including:

* Text classification
* Image recognition
* Sentiment analysis
* Language modeling
* Music generation
* Object detection

Browse our 2M+ models, each trained on a specific dataset and evaluated on performance metrics.

### Datasets

We provide access to millions of datasets, covering topics such as:

* Natural language processing (NLP)
* Computer vision
* Time series analysis
* Sentiment analysis
* Customer feedback
* Healthcare data

Browse our catalog of verified datasets for accurate results and robust performance.

### Spaces

Our feature-rich platform allows users to explore, collaborate, and deploy AI models on various hosting options, including:

* AutoML (Automated Machine Learning)
* Zero-configuration infrastructure
* GPU acceleration
* Cloud providers (AWS, GCP, Azure)

Developers can build scalable applications with ease, by importing their favorite model from our catalog.

## Customers

Our platform is used by top companies and research institutions worldwide, including:

* Google
* Microsoft
* IBM
* Facebook
* LinkedIn

These organizations enable them to collaborate on machine learning projects, train and deploy AI models, and improve decision-making capabilities.

## Careers

We are recruiting talented individuals to join our team in various roles, such as:

* Data Scientist: Develop and maintain high-performance datasets for model training.
* Research Scientist: Collaborate with our community to improve and optimize pre-trained models.
* Developer: Work on deploying AI models and developing new feature-rich platforms.
* Product Manager: Oversee the development of AI business intelligence tools.

Browse our job listings and stay up-to-date with the latest opportunities in AI-driven innovation.

## News & Updates

Stay informed about our recent announcements, product updates, and research milestones. Our platform is constantly evolving to meet the demands of the machine learning community.

* [Recent blog post]
* [GitHub releases with code samples]

## Community

Join our vibrant community of contributors, researchers, and developers who share knowledge, expertise, and enthusiasm for AI innovation.

# Get Started

Sign up for Hugging Face and experience the power of collaborative machine learning. Explore our platform, browse our vast collection of models and datasets, or start building your own applications today!

### Sign Up
[Create a account](https://accounts.huggingface.com/register)

Learn more about our platform: [Community Hub](https://community.huggingface.com)

Stay connected: Follow us on social media and contribute to our ongoing community discussions!

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [39]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="llama3.2:1b",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [40]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2:1b
Found 2 relevant links


# Hugging Face: The AI Community Building the Future

## Introduction

Hugging Face is an AI community building a robust platform for developers, researchers, and users to collaborate on machine learning (ML) models, datasets, and applications. With a vast array of open-source tools and models available, individuals from diverse backgrounds can contribute, share, and build exciting applications.

## About Us

Our mission is to create an inclusive environment where anyone can join the world of AI by providing a scalable, secure, and easy-to-use foundation for ML development. We strive to foster collaboration, innovation, and growth among our users and partners.

## Models

Hugging Face's extensive model library boasts over 2 million+ pre-trained models covering various domains such as:

* Computer Vision: Image recognition, object detection, segmentation
* Natural Language Processing (NLP): Text classification, sentiment analysis, machine translation
* Speech Recognition: Audio-to-text conversion, speech-to-text interaction

## Spaces

We have established a collaborative platform in which users can work on unlimited public models, datasets, and projects. This space offers an ideal environment for sharing ideas, learning from others, and building AI applications.

## Community

Our community is diverse, with members engaging in various activities, including:

* Contributing to model development: By importing pre-trained models or building custom ones, users contribute to the vast repository of Open Data Arena
* Sharing knowledge: With 500k+ datasets, tutorials, and resources available for learning purposes
* Building and deploying AI applications: Using pre-built containers, such as vLLM, TGI, and SGLang
* Participating in community discussions and hackathons to foster innovation

## Career & Recruitment

If you're an aspiring developer or enthusiast looking to join the AI revolution, here's what we have available:

* **Self-Serve Plans**: Flexible pricing options designed for various requirements from individuals to enterprises
* **Enterprise Solutions**: Customized plans tailored to meet specific business needs with dedicated support and security features

Hugging Face Open Jobs:
Explore opportunities at our company as we advance and grow. [Link]

## Getting Started

Want to explore Hugging Face further? Download the catalog of pre-trained models, datasets, and APIs for creating your AI project today! [Open Data Arena Link]

In [41]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2:1b
Found 2 relevant links


## Hugging Face – The AI Community Building the Future

### About Us

Hugging Face is a collaborative platform where the machine learning community comes together to build, discover, and innovate. Our mission is to empower individuals and organizations with access to cutting-edge AI models, datasets, and applications.

### Models

Our platform hosts over 2 million public models, spanning various categories such as Natural Language Processing (NLP), Computer Vision, and Reinforcement Learning. These models are built by community members through collaborative work, using the Hugging Face API. Browse our collection of models to discover new potential applications for your AI projects.

### Datasets

We have created a wide range of datasets, covering various domains and use cases. From text classification and sentiment analysis to computer vision and game development, our datasets provide valuable insights and examples. Explore our dataset offerings to help you develop your AI skills.

### Spaces

Join our thriving community of developers, researchers, and entrepreneurs who are building innovative applications on top of Hugging Face models and datasets. Our spaces offer a platform for members to share knowledge, collaborate on projects, and access exclusive resources.

### Community

Hugging Face is more than just a platform – it's a community of like-minded individuals who share a passion for AI innovation. Join our Discord server or join one of our upcoming Meetups to connect with others, participate in discussions, and stay updated on the latest developments.

### Docs

Our documentation hub provides an exhaustive guide to getting started with Hugging Face models, datasets, and applications. From API guides to model deployment tutorials, we have everything you need to learn and master AI with us.

### Enterprise

Whether you're building a production-ready application or integrating AI into your existing infrastructure, our enterprise solutions are designed to provide security, scalability, and reliability. Explore our offerings for a customized AI experience tailored to your organization's needs.

### Pricing

Our pricing plans align with your budget and usage needs. From pay-per-use models and datasets to our enterprise-grade solutions, we've got you covered. Read more about our pricing options on the [Enterprise page](#pricing).

### Log In / Sign Up

Log in or sign up now to access the full range of Hugging Face features.

#### About
Sergei
Follow
Kalaipriya's profile picture
Renumathi's profile picture
selvivincent's profile picture


## Terms of Service

Hugging Face · GitHub

 Skip to content
Navigation Menu
Toggle navigation
Sign in
Appearance settings
huggingface
Platform
AI CODE CREATION
GitHub Copilot
Write better code with AI
GitHub Spark
Build and deploy intelligent apps
GitHub Models
Manage and compare prompts
MCP Registry
New
Integrate external tools
DEVELOPER WORKFLOWS
Actions
Automate any workflow
Codespaces
Instant dev environments
Issues
Plan and track work
Code Review
Manage code changes
APPLICATION SECURITY
GitHub Advanced Security
Find and fix vulnerabilities
Code security
Secure your code as you build
Secret protection
Stop leaks before they start
EXPLORE
Why GitHub
Documentation
Blog
Changelog
Marketplace
View all features
Solutions
BY COMPANY SIZE
Enterprises
Small and medium teams
Startups
Nonprofits
BY USE CASE
App Modernization
DevSecOps
DevOps
CI/CD
View all use cases
BY INDUSTRY
Healthcare
Financial services
Manufacturing
Government
View all industries
View all solutions
Resources
EXPLORE BY TOPIC
AI
Software Development
DevOps
Security
View all topics
EXPLORE BY TYPE
Customer stories
Events & webinars
Ebooks & reports
Business insights
GitHub Skills
SUPPORT & SERVICES
Documentation
Customer support
Community forum
Trust center
Partners
Open Source
 COMMUNITY
GitHub Sponsors
Fund open source developers
PROGRAMS
Security Lab
Maintainer Community
Accelerator
Archive Program
REPIPORIES
Topics
Trending
Collections
Enterprise
ENTERPRISE SOLUTIONS
Enterprise platform
AI-powered developer platform
AVAILABLE ADD-ONS
GitHub Advanced Security
Enterprise-grade security features
Copilot for Business
Enterprise-grade AI features
Premium Support
Enterprise-grade 24/7 support
Pricing
Search or jump to...
Search code, repositories, users, issues, pull requests...
Search

### Relevance

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>